## 🔹 Part 1 — Data Processing

In [ ]:
# Load the datasets
import pandas as pd
import os

# Define base folder path
DATA_PATH = "store_sales_data"

# Load datasets using os.path.join
train_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"), parse_dates=["date"])
test_df = pd.read_csv(os.path.join(DATA_PATH, "test.csv"), parse_dates=["date"])
store_df = pd.read_csv(os.path.join(DATA_PATH, "stores.csv"))
oil_df = pd.read_csv(os.path.join(DATA_PATH, "oil.csv"), parse_dates=["date"])
transactions_df = pd.read_csv(os.path.join(DATA_PATH, "transactions.csv"), parse_dates=["date"])
holidays_events_df = pd.read_csv(os.path.join(DATA_PATH, "holidays_events.csv"), parse_dates=["date"])


### Check top five rows of each dataframe

In [ ]:

train_df.head(5)

In [ ]:
test_df.head(5)

In [ ]:
transactions_df.head(5)

In [ ]:
store_df.head(5)

In [ ]:
oil_df.head()

In [ ]:
holidays_events_df.head(5)

In [ ]:
# Create a dictionary of your DataFrames
all_dfs = {
    "train": train_df,
    "test": test_df,
    "store": store_df,
    "transactions": transactions_df,
    "oil": oil_df,
    "holidays_events": holidays_events_df
}

# Loop through the dictionary and print the null counts
print("Total Missing Values per DataFrame:")
for name, df in all_dfs.items():
    total_nulls = df.isnull().sum().sum()
    print(f"- {name}_df: {total_nulls}")

In [ ]:
oil_df['dcoilwtico'] = oil_df['dcoilwtico'].interpolate()
oil_df['dcoilwtico'] = oil_df['dcoilwtico'].fillna(method='bfill') # type: ignore

In [ ]:
oil_df.isnull().sum()

In [ ]:
train_df.columns

In [ ]:
train_df.describe()

In [ ]:
len(train_df["family"].unique())

In [ ]:
test_df.describe()

In [ ]:
print(train_df.shape)
print(oil_df.shape)

In [ ]:
train_oil_df = train_df.merge(oil_df, on="date", how="left")

In [ ]:
train_oil_df.isnull().sum().sum()

In [ ]:
train_oil_df.isnull().sum()

In [ ]:
train_oil_df["dcoilwtico"] = train_oil_df["dcoilwtico"].fillna(method="bfill") # type: ignore

In [ ]:
train_oil_df.isnull().sum()

In [ ]:
print(f"train_df shape: {train_df.shape}")
print(f"train_oil_df shape: {train_oil_df.shape}")

In [ ]:
holidays_events_df.head(5)

In [ ]:
holidays_events_df["type"].unique()

In [ ]:
holidays_events_df.shape

In [ ]:
store_df.columns

In [ ]:
train_oil_store_df = train_oil_df.merge(store_df, on='store_nbr', how='left')
train_oil_store_df.head(5)

In [ ]:
train_oil_store_df.isnull().sum().sum()

In [ ]:
holidays_events_df.info()

In [ ]:
# removed tranfereed columns
holidays_events_df = holidays_events_df[holidays_events_df['transferred'] == False]
holidays_events_df.shape

In [ ]:
holidays_events_df['type'].unique()

In [ ]:
holidays_events_df = holidays_events_df[holidays_events_df['type'] != 'Work Day']
holidays_events_df.shape

In [ ]:
train_oil_store_df.head(5)

In [ ]:
holidays_events_df.columns

In [ ]:
holidays_events_df['locale'].unique()

In [ ]:
train_oil_store_holiday_df = train_oil_store_df.merge(holidays_events_df, on='date', how='left')

In [ ]:
train_oil_store_holiday_df.head(5)

In [ ]:
train_oil_store_holiday_df['holiday_flag'] = 0

In [ ]:
train_oil_store_holiday_df.loc[train_oil_store_holiday_df['locale'] == 'National', 'holiday_flag'] = 1

train_oil_store_holiday_df.loc[
    (train_oil_store_holiday_df['locale'] == 'Regional') &
    (train_oil_store_holiday_df['locale_name'] == train_oil_store_holiday_df['state']),
    'holiday_flag'
] = 1

train_oil_store_holiday_df.loc[
    (train_oil_store_holiday_df['locale'] == 'Local') &
    (train_oil_store_holiday_df['locale_name'] == train_oil_store_holiday_df['city']),
    'holiday_flag'
] = 1


In [ ]:
train_oil_store_holiday_df.head(5)

In [ ]:
train_df_final = train_oil_store_holiday_df

In [ ]:
train_df_final.columns

In [ ]:
train_df_final.info()

In [ ]:
holidays_events_df.columns

In [ ]:
train_df_final.drop(["type_y", "type_x", "description", "transferred"], axis=1, inplace=True)

In [ ]:
train_df_final.info()

In [ ]:
train_df_full = train_df_final.merge(transactions_df, on=["store_nbr", "date"], how="left")

In [ ]:
train_df_full.isnull().sum()

In [ ]:
train_df_full.drop(["locale", "locale_name"], axis=1, inplace=True)

In [ ]:
train_df_full.info()

In [ ]:
train_df_full['holiday_flag'].fillna(0, inplace=True)
train_df_full['transactions'].fillna(0, inplace=True)
train_df_full.isnull().sum()

## 🔹 Part 2 — Feature Engineering

In [1]:
import pandas as pd
import os

In [2]:
# Define base folder path
DATA_PATH = "store_sales_data"

In [3]:
# train_df_full.to_csv(os.path.join(DATA_PATH, "train_df_full.csv"), index=False)

In [4]:
train_df_full = pd.read_csv(os.path.join(DATA_PATH, "train_df_full.csv"), parse_dates=["date"])

In [5]:
train_df_full.columns

Index(['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion',
       'dcoilwtico', 'city', 'state', 'cluster', 'holiday_flag',
       'transactions'],
      dtype='object')

In [6]:
train_df_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3052566 entries, 0 to 3052565
Data columns (total 12 columns):
 #   Column        Dtype         
---  ------        -----         
 0   id            int64         
 1   date          datetime64[ns]
 2   store_nbr     int64         
 3   family        object        
 4   sales         float64       
 5   onpromotion   int64         
 6   dcoilwtico    float64       
 7   city          object        
 8   state         object        
 9   cluster       int64         
 10  holiday_flag  int64         
 11  transactions  float64       
dtypes: datetime64[ns](1), float64(3), int64(5), object(3)
memory usage: 279.5+ MB


In [7]:
train_df_full['year']=train_df_full['date'].dt.year
train_df_full['month']=train_df_full['date'].dt.month
train_df_full['day']=train_df_full['date'].dt.day
train_df_full['day_of_week']=train_df_full['date'].dt.dayofweek
train_df_full['week_of_year']=train_df_full['date'].dt.isocalendar().week
train_df_full['is_weekend'] = train_df_full['day_of_week'].isin([5, 6]).astype(int)

In [8]:
train_df_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3052566 entries, 0 to 3052565
Data columns (total 18 columns):
 #   Column        Dtype         
---  ------        -----         
 0   id            int64         
 1   date          datetime64[ns]
 2   store_nbr     int64         
 3   family        object        
 4   sales         float64       
 5   onpromotion   int64         
 6   dcoilwtico    float64       
 7   city          object        
 8   state         object        
 9   cluster       int64         
 10  holiday_flag  int64         
 11  transactions  float64       
 12  year          int32         
 13  month         int32         
 14  day           int32         
 15  day_of_week   int32         
 16  week_of_year  UInt32        
 17  is_weekend    int64         
dtypes: UInt32(1), datetime64[ns](1), float64(3), int32(4), int64(6), object(3)
memory usage: 363.9+ MB


In [9]:
train_sorted_df = train_df_full.sort_values(by=['store_nbr', 'family', 'date'])
lags = [1,7]
for lag in lags:
    train_sorted_df[f'lag_{lag}']=train_sorted_df.groupby(by=['store_nbr', 'family'])['sales'].shift(lag)

In [10]:
grouped_sales = train_sorted_df.groupby(by=['store_nbr', 'family'])['sales']
train_sorted_df['rolling_mean_7'] = grouped_sales.transform( lambda x: x.shift(1).rolling(window=7).mean())
train_sorted_df['rolling_var_7'] = grouped_sales.transform( lambda x: x.shift(1).rolling(window=7).std())

In [11]:
grouped_promo = train_sorted_df.groupby(by=['store_nbr', 'family'])['onpromotion']
train_sorted_df['promo_rolling_mean_7'] = grouped_promo.transform(lambda x: x.shift(1).rolling(window=7).mean())

In [12]:
train_sorted_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3052566 entries, 0 to 3052433
Data columns (total 23 columns):
 #   Column                Dtype         
---  ------                -----         
 0   id                    int64         
 1   date                  datetime64[ns]
 2   store_nbr             int64         
 3   family                object        
 4   sales                 float64       
 5   onpromotion           int64         
 6   dcoilwtico            float64       
 7   city                  object        
 8   state                 object        
 9   cluster               int64         
 10  holiday_flag          int64         
 11  transactions          float64       
 12  year                  int32         
 13  month                 int32         
 14  day                   int32         
 15  day_of_week           int32         
 16  week_of_year          UInt32        
 17  is_weekend            int64         
 18  lag_1                 float64       
 19  lag_7

In [13]:
li = ['cluster','state', 'store_nbr', 'city', 'family']
for i in li:
    print(len(train_sorted_df[i].unique()))

17
16
54
22
33


In [14]:
from sklearn.preprocessing import LabelEncoder

In [15]:
encoder = {}
for l in li:
    le = LabelEncoder()
    train_sorted_df[f"{l}_encoded"] = le.fit_transform(train_sorted_df[l])
    encoder[l] = le
    train_sorted_df.drop(l, axis=1, inplace=True)

In [16]:
import pickle
file_name = 'LabelEncoder.pkl'
with open(file_name, 'wb') as file:
    pickle.dump(encoder, file)

In [17]:
train_sorted_df.to_csv(os.path.join(DATA_PATH, "df_final.csv"), index=False)

## 🔹 Part 3 — Model Training

In [ ]:
train_df_final = pd.read_csv("")